In [4]:
using Random
using RobustNeuralNetworks
using Flux
using CUDA
using FileIO
using Images
using ImageMagick
using ImageTransformations  # For image resizing


In [58]:

# Random seed for consistency
rng = MersenneTwister(42)
print
# Model specification
nu = 64 * 64 * 3  # Number of inputs (size of flattened image)
ny = 2             # Number of outputs (human vs non-human classification)
nh = fill(64,4)# Two hidden layers with 128 and 64 neurons respectively
γ  = 20.0f0         # Lipschitz bound of 5.0


20.0f0

In [59]:
# Define model parameters and architecture
model_ps = DenseLBDNParams{Float32}(nu, nh, ny, γ; rng)
model = Chain(DiffLBDN(model_ps)) |> gpu  # No softmax, since we use logitcrossentropy


Chain(
  DiffLBDN(
    DenseLBDNParams(
      DirectLBDNParams{Float32, 5, 4}((Float32[0.015360689 0.0017208339 … -0.009691107 -0.0015726191; -0.0010026366 -0.013676248 … -0.008386618 -0.006045376; … ; -0.029143455 -0.007263975 … -0.0056053344 -0.011105277; -0.01421171 0.005446661 … 0.012590497 -0.0031926206], Float32[-0.0960662 0.14747265 … 0.021691943 0.079210036; 0.074275464 -0.02377779 … 0.042318456 -0.059942834; … ; 0.07678836 -0.053540703 … -0.09182376 0.094155505; -0.1374687 0.059249926 … -0.11002277 0.06810896], Float32[0.08448361 0.09400149 … 0.109772556 -0.056795023; 0.043846942 -0.07523433 … -0.046458215 0.0036239782; … ; -0.0067473208 -0.10242455 … -0.026694076 0.047516968; 0.21325561 0.053234693 … 0.25353712 -0.02929036], Float32[-0.013370613 0.02661627 … -0.08371441 0.13863544; 0.06869661 -0.12131813 … 0.04204886 -0.029203784; … ; 0.002314436 -0.04222187 … 0.09494239 -0.106546946; 0.08806745 -0.111679345 … 0.044569265 0.02880246], Float32[0.14077823 0.06107566; 0.06965737

In [7]:

# Function to load images from a folder
function load_images_from_folder(folder_path)
    images = []
    supported_extensions = [".jpg", ".jpeg", ".png", ".bmp", ".gif"]
    for file in readdir(folder_path)
        ext = lowercase(splitext(file)[2])
        if ext in supported_extensions
            img_path = joinpath(folder_path, file)
            img = load(img_path)
            push!(images, img)
        end
    end
    return images
end

load_images_from_folder (generic function with 1 method)

In [8]:
# Function to preprocess images
function preprocess_images(images, target_size=(64, 64))
    return [reshape(channelview(imresize(img, target_size)), :) ./ 255.0 for img in images]
end


preprocess_images (generic function with 2 methods)

In [9]:

# Paths to image folders
human_images_path = "C:\\Users\\jonat\\OneDrive\\Desktop\\Human_vs_non_human\\human-and-non-human\\versions\\1\\human-and-non-human\\training_set\\training_set\\humans"
non_human_images_path = "C:\\Users\\jonat\\OneDrive\\Desktop\\Human_vs_non_human\\human-and-non-human\\versions\\1\\human-and-non-human\\training_set\\training_set\\non-humans"

# Load and preprocess images
human_images = preprocess_images(load_images_from_folder(human_images_path))
non_human_images = preprocess_images(load_images_from_folder(non_human_images_path))

# Prepare labels
human_labels = [1 for _ in 1:length(human_images)]
non_human_labels = [0 for _ in 1:length(non_human_images)]


4006-element Vector{Int64}:
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 ⋮
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0

In [10]:

# Combine images and labels
images = vcat(human_images, non_human_images)
labels = vcat(human_labels, non_human_labels)

# Convert labels to one-hot encoded vectors
y_train = Flux.onehotbatch(labels, 0:1)


2×8017 OneHotMatrix(::Vector{UInt32}) with eltype Bool:
 ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  …  1  1  1  1  1  1  1  1  1  1  1  1
 1  1  1  1  1  1  1  1  1  1  1  1  1     ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅

In [ ]:
X_train = cat(images..., dims=2) |> gpu  # Stack images along second dimension
Y_train = y_train |> gpu

2×8017 OneHotMatrix(::CuArray{UInt32, 1, CUDA.DeviceMemory}) with eltype Bool:
 ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  …  1  1  1  1  1  1  1  1  1  1  1  1
 1  1  1  1  1  1  1  1  1  1  1  1  1     ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅

In [13]:

# Prepare data for training
data = [(X_train, Y_train)]

1-element Vector{Tuple{CuArray{Float32, 2, CUDA.DeviceMemory}, OneHotArrays.OneHotMatrix{UInt32, CuArray{UInt32, 1, CUDA.DeviceMemory}}}}:
 ([0.0 0.0 … 0.002675894 0.0032602844; 0.0 0.0 … 0.0026143792 0.0032602844; … ; 0.002522107 0.0014148405 … 0.0013840831 0.0038754325; 0.0018300654 0.0015993848 … 0.0014302192 0.0038600538], Bool[0 0 … 1 1; 1 1 … 0 0])

In [14]:
using Flux.Optimisers: Adam
using Flux.Losses: logitcrossentropy

In [15]:

optimizer = Descent()
loss(model,x,y) = Flux.logitcrossentropy(model(x), y)


loss (generic function with 1 method)

In [16]:
using Statistics
using Flux:OneHotMatrix
# Check test accuracy during training
compare(y::OneHotMatrix, ŷ) = maximum(ŷ, dims=1) .== maximum(y.*ŷ, dims=1)
accuracy(model, x, y::OneHotMatrix) = mean(compare(y, model(x)))

# Callback function to show results while training
function progress(model, iter,a,b)
    train_loss = round(loss(model, Flux.flatten(a), b), digits=4)
    #test_acc = round(accuracy(model, x_test, y_test), digits=4)
    @show iter train_loss #test_acc
    println()
end


progress (generic function with 1 method)

In [18]:
function train_model(data, epochs,opt,loss1,mod)
    for epoch in 1:epochs
        Flux.train!(loss1, model, data, opt)|>gpu
        println("Epoch $epoch completed")
        #progress(model)
        progress(model,epoch,X_train,Y_train)
    end
end




train_model (generic function with 1 method)

In [60]:
using BSON



opt_state = Flux.setup(Adam(0.0058), model)
train_model(data, 100,opt_state,loss,model)  # Train for 10 epochs

bson("well_i_try_a_ting5.bson", Dict("model" => model |> cpu))


Epoch 1 completed
iter = 1
train_loss = 3.7411f0

Epoch 2 completed
iter = 2
train_loss = 0.6978f0

Epoch 3 completed
iter = 3
train_loss = 1.2329f0

Epoch 4 completed
iter = 4
train_loss = 0.6817f0

Epoch 5 completed
iter = 5
train_loss = 0.7782f0

Epoch 6 completed
iter = 6
train_loss = 0.6737f0

Epoch 7 completed
iter = 7
train_loss = 0.7097f0

Epoch 8 completed
iter = 8
train_loss = 0.6677f0

Epoch 9 completed
iter = 9
train_loss = 0.6579f0

Epoch 10 completed
iter = 10
train_loss = 0.6449f0

Epoch 11 completed
iter = 11
train_loss = 0.6186f0

Epoch 12 completed
iter = 12
train_loss = 0.6172f0

Epoch 13 completed
iter = 13
train_loss = 0.5851f0

Epoch 14 completed
iter = 14
train_loss = 0.5839f0

Epoch 15 completed
iter = 15
train_loss = 0.5602f0

Epoch 16 completed
iter = 16
train_loss = 0.5597f0

Epoch 17 completed
iter = 17
train_loss = 0.5407f0

Epoch 18 completed
iter = 18
train_loss = 0.53f0

Epoch 19 completed
iter = 19
train_loss = 0.53f0

Epoch 20 completed
iter = 20
train

In [64]:
train_acc = accuracy(model, X_train, Y_train)*100


86.61594112510915

In [63]:
test_acc = accuracy(model, X_test, Y_test)*100


82.0051413881748

2×2723 OneHotMatrix(::CuArray{UInt32, 1, CUDA.DeviceMemory}) with eltype Bool:
 ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  …  1  1  1  1  1  1  1  1  1  1  1  1
 1  1  1  1  1  1  1  1  1  1  1  1  1     ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅

In [ ]:
# Paths to image folders
human_images_path1 ="C:\\Users\\jonat\\OneDrive\\Desktop\\Human_vs_non_human\\human-and-non-human\\versions\\1\\human-and-non-human\\test_set\\test_set\\humans"
non_human_images_path1 = "C:\\Users\\jonat\\OneDrive\\Desktop\\Human_vs_non_human\\human-and-non-human\\versions\\1\\human-and-non-human\\test_set\\test_set\\non-humans"

# Load and preprocess images
human_images1 = preprocess_images(load_images_from_folder(human_images_path1))
non_human_images1 = preprocess_images(load_images_from_folder(non_human_images_path1))

# Prepare labels
human_labels1 = [1 for _ in 1:length(human_images1)]
non_human_labels1 = [0 for _ in 1:length(non_human_images1)]


# Combine images and labels
images1 = vcat(human_images1, non_human_images1)
labels1 = vcat(human_labels1, non_human_labels1)
X_test = cat(images1..., dims=2) |> gpu  # Stack images along second dimension

# Convert labels to one-hot encoded vectors
y_test = Flux.onehotbatch(labels1, 0:1)
Y_test = y_test |> gpu



2×8017 OneHotMatrix(::CuArray{UInt32, 1, CUDA.DeviceMemory}) with eltype Bool:
 ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  …  1  1  1  1  1  1  1  1  1  1  1  1
 1  1  1  1  1  1  1  1  1  1  1  1  1     ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅  ⋅

┌ Warning: Found `resolution` in the theme when creating a `Scene`. The `resolution` keyword for `Scene`s and `Figure`s has been deprecated. Use `Figure(; size = ...` or `Scene(; size = ...)` instead, which better reflects that this is a unitless size and not a pixel resolution. The key could also come from `set_theme!` calls or related theming functions.
└ @ Makie C:\Users\jonat\.julia\packages\Makie\Q6F2P\src\scenes.jl:238


LoadError: DimensionMismatch: new dimensions (64, 64) must be consistent with array size 12288

In [ ]:

# Define Gaussian noise function
gaussian(x) = randn(rng, T, size(x)...) |> dev

# Function to compute test accuracy with Gaussian noise
function noisy_test_error_gaussian(model, σ=0)
    noisy_xtest = x_test .+ σ * gaussian(x_test)
    accuracy(model, noisy_xtest, y_test) * 100
end

# Define noise levels
σs = T.(LinRange(0, 50, 10)) ./ 255

# Compute errors for both models
lbdn_error_gaussian = noisy_test_error_gaussian.((model,), σs)
dense_error_gaussian = noisy_test_error_gaussian.((dense,), σs)

# Plot results
fig = Figure(resolution=(500,300))
ax1 = Axis(fig[1,1], xlabel="Gaussian Noise Level (σ)", ylabel="Test accuracy (%)")
lines!(ax1, σs, lbdn_error_gaussian, label="LBDN (γ=5)")
lines!(ax1, σs, dense_error_gaussian, label="Dense")

xlims!(ax1, 0, 0.2)
axislegend(ax1, position=:lb)
save("lbdn_gaussian_robust.svg", fig)
